<a href="https://colab.research.google.com/github/ruantuan1/Video-Game-Sales-Risk-Prediction/blob/main/Video_Game_Sales_Risk_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import pandas as pd

df = pd.read_csv('/content/vgsales.csv')
df.head(10)

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37
5,6,Tetris,GB,1989.0,Puzzle,Nintendo,23.20,2.26,4.22,0.58,30.26
6,7,New Super Mario Bros.,DS,2006.0,Platform,Nintendo,11.38,9.23,6.50,2.90,30.01
7,8,Wii Play,Wii,2006.0,Misc,Nintendo,14.03,9.20,2.93,2.85,29.02
8,9,New Super Mario Bros. Wii,Wii,2009.0,Platform,Nintendo,14.59,7.06,4.70,2.26,28.62
9,10,Duck Hunt,NES,1984.0,Shooter,Nintendo,26.93,0.63,0.28,0.47,28.31


In [8]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  object 
 2   Platform      16598 non-null  object 
 3   Year          16327 non-null  float64
 4   Genre         16598 non-null  object 
 5   Publisher     16540 non-null  object 
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: float64(6), int64(1), object(4)
memory usage: 1.4+ MB


,0
Rank,0
Name,0
Platform,0
Year,271
Genre,0
Publisher,58
NA_Sales,0
EU_Sales,0
JP_Sales,0
Other_Sales,0


In [33]:
#Xóa các dòng year = null và chuyển null Puibliser thành unknown
df = df.dropna(subset=['Year'])
df['Publisher'] = df['Publisher'].fillna('Unknown')
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
Index: 16327 entries, 0 to 16597
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16327 non-null  int64  
 1   Name          16327 non-null  object 
 2   Platform      16327 non-null  object 
 3   Year          16327 non-null  float64
 4   Genre         16327 non-null  object 
 5   Publisher     16327 non-null  object 
 6   NA_Sales      16327 non-null  float64
 7   EU_Sales      16327 non-null  float64
 8   JP_Sales      16327 non-null  float64
 9   Other_Sales   16327 non-null  float64
 10  Global_Sales  16327 non-null  float64
 11  Hit           16327 non-null  int64  
dtypes: float64(6), int64(2), object(4)
memory usage: 1.6+ MB


,0
Rank,0
Name,0
Platform,0
Year,0
Genre,0
Publisher,0
NA_Sales,0
EU_Sales,0
JP_Sales,0
Other_Sales,0


In [44]:
#Tạo cột hit (nếu Global sale < 1m bản bán thì 1 còn lại 0)
df['Hit'] = (df['Global_Sales'] < 1).astype(int)
df[['Global_Sales', 'Hit']].head(10)

,Global_Sales,Hit
0,82.74,0
1,40.24,0
2,35.82,0
3,33.00,0
4,31.37,0
5,30.26,0
6,30.01,0
7,29.02,0
8,28.62,0
9,28.31,0


In [45]:
df.value_counts('Hit') #Database bị lệch

,count
Hit,
1,14268
0,2059


In [25]:
print(df.columns.tolist())

['Rank', 'Name', 'Platform', 'Year', 'Genre', 'Publisher', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales', 'Hit']


In [46]:
features = ['Platform', 'Year', 'Genre', 'Publisher']
X = df.loc[:, features] #Dữ liệu đầu ào
y = df['Hit']           #Nhãn cần dự đoán

In [47]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ['Platform', 'Genre', 'Publisher']
numeric_features = ['Year']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numeric_features)
    ]
)

In [48]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [49]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight='balanced'
    ))
])

In [50]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Platform', 'Genre',
                                                   'Publisher']),
                                                 ('num', 'passthrough',
                                                  ['Year'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced',
                                        n_estimators=200, random_state=42))])

In [51]:
y_pred = model.predict(X_test)

In [52]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

0.8447642375995101

In [53]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, y_pred)

array([[ 179,  206],
       [ 301, 2580]])

In [54]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.37      0.46      0.41       385
           1       0.93      0.90      0.91      2881

    accuracy                           0.84      3266
   macro avg       0.65      0.68      0.66      3266
weighted avg       0.86      0.84      0.85      3266



In [55]:
import pandas as pd

feature_names = model.named_steps['preprocessor'].get_feature_names_out()
importances = model.named_steps['classifier'].feature_importances_

feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

feat_imp.head(15)

,feature,importance
573,num__Year,0.277042
375,cat__Publisher_Nintendo,0.057205
171,cat__Publisher_Electronic Arts,0.029788
31,cat__Genre_Action,0.021469
41,cat__Genre_Sports,0.018832
32,cat__Genre_Adventure,0.017880
63,cat__Publisher_Activision,0.017443
39,cat__Genre_Shooter,0.014937
38,cat__Genre_Role-Playing,0.013855
16,cat__Platform_PS2,0.013824


Year chiếm 28%, còn lại là Publisher + Genre + Platform
Năm phát hành ảnh hướng rất nhiều đến việc game bán chạy hay không
*   Game ra quá sớm (thị trường nhỏ)
*   Game ra quá muộn (platform thoái trào)
*   Game ra ngoài “thời kỳ vàng”

Ra game sai thời điểm -> rủi ro rất cao







Các nhà phát hành lớn:
*   Nintendo
*   EA
*   Activison
*   Ubisoft
*   Sega

NPH lớn phát hành rất nhiều game, trong đó có cả game fail và model học được pattern rủi ro trong danh mục của họ

->NPH lớn không đảm bảo thành công


Các thể loại game nổi bật:
*   Action
*   Sport
*   Adventure
*   Shooting
*   RPG

Đây là các thể loại có tính cạnh tranh cao, có rất nhiều game -> thị trường bão hòa, nhiều game trung bình nếu không nỗi bật sẽ thất bại ngay lập tức.

-> Phát hành game thuộc thể loại phổ biến cũng không an toán và không đảm bảo game sẽ bán được.

Flatform (Nền tảng game)
*   PS2
*   DS

Đây là các nền tảng game đã đi tới cuối vòng đởi và người chơi bắt đầu chuyển sang các nền tảng game khác như PS3, XBOX 360

-> Ra game mà không theo Flatform hiện thời thì tỉ lệ game không bán được rất cao

## TỔNG HỢP
Game sẽ không bán chạy nếu :
1.   Ra mắt sai thời.
2.   Phát hành trên platform thoái trào.
3.   Thuộc genre cạnh tranh khốc liệt.
4.   Dù publisher có lớn đến đâu.

Year là vua

Publisher & Genre là bộ lọc rủi ro

Platform là yếu tố timing

In [56]:
#Kiểm tra model
sample = X_test.iloc[[0]]
sample

,Platform,Year,Genre,Publisher
5468,GC,2005.0,Simulation,Electronic Arts


In [57]:
#Model dự đoán game trên không hit
model.predict(sample)

array([1])

In [58]:
model.predict_proba(sample)

array([[0.0474726, 0.9525274]])

In [67]:
#Tối ưu model, chỉnh threshold lên 0.
y_proba = model.predict_proba(X_test)

# Class 1 = Not Hit
y_pred_custom = (y_proba[:, 1] >= 0.7).astype(int)

In [68]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_custom))

              precision    recall  f1-score   support

           0       0.33      0.58      0.42       385
           1       0.94      0.85      0.89      2881

    accuracy                           0.81      3266
   macro avg       0.64      0.71      0.66      3266
weighted avg       0.87      0.81      0.83      3266



In [69]:
def predict_game_risk(platform, genre, publisher, year, threshold=0.7):
    import pandas as pd

    input_df = pd.DataFrame([{
        'Platform': platform,
        'Genre': genre,
        'Publisher': publisher,
        'Year': year
    }])

    proba = model.predict_proba(input_df)[0]
    not_hit_prob = proba[1]   # Class 1 = Not Hit

    if not_hit_prob >= threshold:
        risk = "HIGH"
        decision = "DO NOT INVEST"
    elif not_hit_prob >= 0.5:
        risk = "MEDIUM"
        decision = "REVIEW CAREFULLY"
    else:
        risk = "LOW"
        decision = "SAFE TO PROCEED"

    return {
        "Not_Hit_Probability": round(not_hit_prob, 2),
        "Risk_Level": risk,
        "Decision": decision
    }


In [76]:
predict_game_risk(
    platform="NES",
    genre="Platform",
    publisher="Nintendo",
    year=1985
)

{'Not_Hit_Probability': np.float64(0.16),
 'Risk_Level': 'LOW',
 'Decision': 'SAFE TO PROCEED'}